# BLDC 12S14P — ML-подбор размеров ротора

Интерфейс использует суррогатную модель, обученную на 144 расчётах FEMM.

Область применимости:

- статор: 12 пазов, Ø38,5 мм;
- ротор: 14 полюсов;
- 20 витков на зуб;
- зазор: 0,50–1,50 мм;
- толщина магнита: 1,50–3,00 мм;
- ток: 1–3 А.

Программа не заменяет механическую, тепловую и экспериментальную проверку.

In [ ]:
!pip -q install gradio
import io, joblib
import numpy as np
import pandas as pd
import gradio as gr
from google.colab import files
print('Загрузите BLDC_torque_ML_model.joblib')
uploaded = files.upload()
model_file = next(name for name in uploaded if name.endswith('.joblib'))
model = joblib.load(io.BytesIO(uploaded[model_file]))
print('ML-модель загружена')

In [ ]:
FEATURES = ['gap_mm', 'magnet_thickness_mm', 'current_A']
STATOR_OD_MM = 38.50
ROTOR_YOKE_RADIAL_MM = 2.80

def recommend(current_A, minimum_safe_gap_mm, maximum_magnet_thickness_mm):
    current_A = float(current_A)
    minimum_safe_gap_mm = float(minimum_safe_gap_mm)
    maximum_magnet_thickness_mm = float(maximum_magnet_thickness_mm)

    if not 1.0 <= current_A <= 3.0:
        return 'Ошибка: ток должен быть от 1 до 3 А.'
    if not 0.50 <= minimum_safe_gap_mm <= 1.50:
        return 'Ошибка: минимальный зазор должен быть от 0,50 до 1,50 мм.'
    if not 1.50 <= maximum_magnet_thickness_mm <= 3.00:
        return 'Ошибка: толщина магнита должна быть от 1,50 до 3,00 мм.'

    gaps = np.arange(minimum_safe_gap_mm, 1.5001, 0.005)
    thicknesses = np.arange(1.50, maximum_magnet_thickness_mm + 0.0001, 0.01)
    search = pd.MultiIndex.from_product(
        [gaps, thicknesses, [current_A]], names=FEATURES
    ).to_frame(index=False)
    search['torque_mNm'] = model.predict(search[FEATURES])
    best = search.loc[search.torque_mNm.idxmax()]

    magnet_inner_d = STATOR_OD_MM + 2 * best.gap_mm
    magnet_seat_d = magnet_inner_d + 2 * best.magnet_thickness_mm
    rotor_od = magnet_seat_d + 2 * ROTOR_YOKE_RADIAL_MM

    return (
        f'РЕКОМЕНДУЕМЫЕ ПАРАМЕТРЫ\n\n'
        f'Воздушный зазор: {best.gap_mm:.3f} мм\n'
        f'Толщина магнита: {best.magnet_thickness_mm:.3f} мм\n'
        f'Ток: {best.current_A:.2f} А\n\n'
        f'Внутренний диаметр по магнитам: {magnet_inner_d:.3f} мм\n'
        f'Диаметр посадки магнитов: {magnet_seat_d:.3f} мм\n'
        f'Наружный диаметр ротора: {rotor_od:.3f} мм\n\n'
        f'Прогнозируемый момент: {best.torque_mNm:.3f} мН·м'
    )

demo = gr.Interface(
    fn=recommend,
    inputs=[
        gr.Slider(1.0, 3.0, value=2.0, step=0.1, label='Ток, А'),
        gr.Slider(0.50, 1.50, value=0.95, step=0.05, label='Минимальный безопасный зазор, мм'),
        gr.Slider(1.50, 3.00, value=2.00, step=0.10, label='Максимальная доступная толщина магнита, мм')
    ],
    outputs=gr.Textbox(label='Рекомендация ML', lines=11),
    title='ML-проектирование 3D-печатного BLDC 12S14P',
    description='Подбор размеров ротора для статора Ø38,5 мм по результатам 144 FEMM-расчётов.',
    examples=[[2.0, 0.95, 2.0], [2.0, 0.50, 2.0], [3.0, 0.70, 3.0]]
)

demo.launch(share=False, debug=False)